In [1]:
import pandas as pd
import numpy as np
import re
import csv
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from sklearn.feature_extraction.text import TfidfVectorizer

print("="*60)
print("INITIALISIERUNG DER ENVIRONMENT")
print("="*60)



# NLTK Stoppwörter geräuschlos prüfen und nur bei Bedarf laden
try:
    nltk.data.find('corpora/stopwords')
    print("-> NLTK Stoppwörter bereits lokal vorhanden (Download übersprungen).")
except LookupError:
    print("-> NLTK Stoppwörter nicht gefunden. Starte Download...")
    nltk.download('stopwords')

english_stop_words = stopwords.words('english')
print("-> Bibliotheken erfolgreich initialisiert.")
print("="*60 + "\n")

# ==========================================
# 1. DATA LOADING (SEMIKOLON-DATENSATZ)
# ==========================================
file_path = "data/Comcast.csv"  # Achte darauf, dass der Ordnerpfad stimmt!

try:
    df = pd.read_csv(file_path, sep=';', encoding='utf-8')
    print("="*60)
    print("DATENSATZ ERFOLGREICH GELADEN")
    print("="*60)
    print(f"Anzahl geladener Roh-Einträge: {len(df)}")
    print("Verfügbare Spalten in der Datei:")
    print(df.columns.tolist())
    print("="*60 + "\n")
except Exception as e:
    print(f"Fehler beim Laden der Datei: {e}")

# Sicherstellen, dass keine leeren Zeilen analysiert werden
df = df.dropna(subset=['Customer Complaint'])



INITIALISIERUNG DER ENVIRONMENT
-> NLTK Stoppwörter bereits lokal vorhanden (Download übersprungen).
-> Bibliotheken erfolgreich initialisiert.

DATENSATZ ERFOLGREICH GELADEN
Anzahl geladener Roh-Einträge: 2224
Verfügbare Spalten in der Datei:
['Ticket #', 'Customer Complaint', 'Date', 'Date_month_year', 'Time', 'Received Via', 'City', 'State', 'Zip code', 'Status', 'Filing on Behalf of Someone']



In [2]:
# ==========================================
# 2. TEXT-VORVERARBEITUNG (PREPROCESSING)
# ==========================================
# NLTK Lemmatizer initialisieren
nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    
    words = text.split()
    
    cleaned_words = []
    for w in words:
        if w not in english_stop_words:
           
            lemma = lemmatizer.lemmatize(w, pos=wordnet.VERB)
           
            lemma = lemmatizer.lemmatize(lemma, pos=wordnet.NOUN)
            cleaned_words.append(lemma)
            
    return " ".join(cleaned_words)
# Preprocessing auf die Spalte anwenden
df['Sauberer_Text'] = df['Customer Complaint'].apply(clean_text)

print("="*60)
print("TEXT-VORVERARBEITUNG (PREPROCESSING)")
print("="*60)
print("Beispiel-Beschwerde VORHER:")
print(f"'{df['Customer Complaint'].iloc[3]}'")
print("\nBeispiel-Beschwerde NACHHER (Bereinigt & Lowercase):")
print(f"'{df['Sauberer_Text'].iloc[3]}'")
print("="*60 + "\n")



[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\chris\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\chris\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


TEXT-VORVERARBEITUNG (PREPROCESSING)
Beispiel-Beschwerde VORHER:
'Comcast Imposed a New Usage Cap of 300GB that punishes streaming.'

Beispiel-Beschwerde NACHHER (Bereinigt & Lowercase):
'comcast impose new usage cap gb punish stream'



In [3]:
# ==========================================
# 3. VEKTORISIERUNG (MATH. TRANSFORMATION)
# ==========================================
# 1. Domänenspezifische Stoppwörter definieren
custom_stop_words = set(stopwords.words('english'))
domain_stops = {
    'comcast', 'service', 'issue', 'problem', 'get', 'call', 
    'would', 'one', 'like', 'back', 'company', 'day', 'time',
    'said', 'told', 'even', 'go', 'know', 'also', 'never', 'complaint'
}
custom_stop_words.update(domain_stops)

# 2. Vectorizer mit strengerem max_df und custom_stop_words anpassen
# max_df=0.4 -> Ignoriert Wörter, die in mehr als 40% der Dokumente vorkommen (filtert das Rauschen!)
count_vectorizer = CountVectorizer(
    max_df=0.8, 
    min_df=3, 
    stop_words=list(custom_stop_words)
)

tf_vectorizer = TfidfVectorizer(
    max_df=0.8, 
    min_df=3, 
    stop_words=list(custom_stop_words)
)
count_data = count_vectorizer.fit_transform(df['Sauberer_Text'])
tfidf_data = tf_vectorizer.fit_transform(df['Sauberer_Text'])

print("="*60)
print("ERKLÄRUNG DER DATENDIMENSIONEN (MATRIZEN)")
print("="*60)
print(f"Form der Count-Matrix: {count_data.shape}")
print(f"Form der TF-IDF-Matrix: {tfidf_data.shape}")
print(f"-> Das bedeutet: Es wurden {count_data.shape[0]} Beschwerdetexte erfolgreich verarbeitet.")
print(f"-> Nach der Bereinigung verbleiben exakt {count_data.shape[1]} einzigartige, relevante Wörter.")
print("="*60 + "\n")


ERKLÄRUNG DER DATENDIMENSIONEN (MATRIZEN)
Form der Count-Matrix: (2224, 327)
Form der TF-IDF-Matrix: (2224, 327)
-> Das bedeutet: Es wurden 2224 Beschwerdetexte erfolgreich verarbeitet.
-> Nach der Bereinigung verbleiben exakt 327 einzigartige, relevante Wörter.



In [4]:
# ==========================================
# 4. THEMENEXTRAKTION (TOPIC MODELING)
# ==========================================
n_topics = 5  # Anzahl der zu extrahierenden Themen

# Modell A: LDA (Latent Dirichlet Allocation)
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_topics = lda.fit_transform(count_data)

# Modell B: NMF (Non-Negative Matrix Factorization)
nmf = NMF(n_components=n_topics, random_state=42)
nmf_topics = nmf.fit_transform(tfidf_data)


In [5]:
# ==========================================
# 5. DYNAMISCHE & STRUKTURIERTE AUSGABE
# ==========================================
def print_dynamische_themen(model, model_topic_matrix, feature_names, n_top_words, model_name):
    print("#"*80)
    print(f"  {model_name} - Reine Modell-Ergebnisse (Rein dynamisch)")
    print("#"*80)
    
    for topic_idx, topic in enumerate(model.components_):
        # 1. Extrahiere alle Top-Wörter
        top_words_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words_list = [feature_names[i] for i in top_words_indices]
        themen_strings = ", ".join(top_words_list)
        
   
        print(f"\nTHEMA #{topic_idx + 1}")
        print(f"-> Extrahierte Top-Wörter: {themen_strings}")
        
            
    print("\n" + "#"*80 + "\n")

# Rein dynamische Ausgabe der Ergebnisse ohne Hardcoding
print_dynamische_themen(lda, lda_topics, count_vectorizer.get_feature_names_out(), 6, "LDA (Latent Dirichlet Allocation)")
print_dynamische_themen(nmf, nmf_topics, tf_vectorizer.get_feature_names_out(), 6, "NMF (Non-Negative Matrix Factorization)")

################################################################################
  LDA (Latent Dirichlet Allocation) - Reine Modell-Ergebnisse (Rein dynamisch)
################################################################################

THEMA #1
-> Extrahierte Top-Wörter: data, cap, charge, practice, usage, unfair

THEMA #2
-> Extrahierte Top-Wörter: internet, speed, throttle, price, connection, pay

THEMA #3
-> Extrahierte Top-Wörter: internet, cable, business, overcharge, outage, phone

THEMA #4
-> Extrahierte Top-Wörter: bill, customer, price, slow, poor, xfinity

THEMA #5
-> Extrahierte Top-Wörter: bill, comcastxfinity, advertise, false, contract, switch

################################################################################

################################################################################
  NMF (Non-Negative Matrix Factorization) - Reine Modell-Ergebnisse (Rein dynamisch)
###############################################################################

In [6]:
# ==========================================
# 6. EVALUATION DER THEMENKOHÄRENZ (COHERENCE)
# ==========================================
# Für die Berechnung nutzen wir gensim oder eine vereinfachte Coherence-Metrik


# Vorbereitung der Tokenized Texts
texts = [text.split() for text in df['Sauberer_Text']]
dictionary = Dictionary(texts)

# Extraktion der Top-Wörter für das NMF-Modell als Liste
feature_names = tf_vectorizer.get_feature_names_out()
nmf_top_words = []
for topic in nmf.components_:
    top_words = [feature_names[i] for i in topic.argsort()[:-10:-1]]
    nmf_top_words.append(top_words)

# Berechnung des Coherence Scores (C_v)
cm = CoherenceModel(topics=nmf_top_words, texts=texts, dictionary=dictionary, coherence='c_v')
coherence_score = cm.get_coherence()

print("="*60)
print(f"MATHEMATISCHE EVALUATION DER THEMENKOHÄRENZ")
print("="*60)
print(f"NMF Topic Coherence Score (C_v): {coherence_score:.4f}")
print("-> Ein höherer Coherence Score bestätigt eine mathematisch valide,")
print("   semantische Zusammengehörigkeit der extrahierten Begriffe.")
print("="*60)

MATHEMATISCHE EVALUATION DER THEMENKOHÄRENZ
NMF Topic Coherence Score (C_v): 0.3957
-> Ein höherer Coherence Score bestätigt eine mathematisch valide,
   semantische Zusammengehörigkeit der extrahierten Begriffe.
